In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 18
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 18
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
!bash start_data_swarm.sh

In [ ]:
!bash stop_data_swarm.sh

In [ ]:
!python -m _tools.check_and_heal_data

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
#Запуск нескольких процессов в параллели
!bash start_swarm.sh \
    --dataset_dir "data/processed/2000_2026_1d_18_3" \
    --bonus_ratio 0.2 \
    --min_delta 0.001 \
    --runs 400 --epochs 100 \
    --lr 2e-3 --l2_reg 1e-5 \
    --start_fold "fold_2013" \
    --factor 0.5 --patience 3 \
    --vram 10000/8 --stagger 0 \
    --keep 5 \
    --append \
    --arch mlp
    #--arch cnn, conv1d+gru, mlp, attention
    #--track_trajectory
    #--init_pca_coord -148.0 -116.0 --init_pca_radius 20.0 \

In [1]:
!./stop_swarm.sh
!python -m _tools.clean_lstm_models --keep 5

In [ ]:
#python run_all_ensembles.py --dataset_dir "data/processed/2000_2026_1d_10_1" --max_k 20
import os
from pathlib import Path
DATASET_DIR = "data/processed/2000_2026_1d_5_10"
MAX_K = 20
folds = sorted([d.name for d in Path(DATASET_DIR).glob("fold_*") if d.is_dir()])
print(f"🔍 Найдено фолдов: {len(folds)} в конфигурации {DATASET_DIR}\n")
for fold in folds:
    print("\n" + "="*80)
    print(f"🚀 ЗАПУСК АНСАМБЛИРОВАНИЯ: {fold}")
    print("="*80)
    !python -m _tools.ensemble_predictor --dataset_dir {DATASET_DIR} --fold {fold} --max_k {MAX_K}

In [ ]:
%run _tools/plot_landscape.py --fold data/processed/2000_2026_1d_30_5/fold_2014

In [ ]:
%matplotlib inline
%run _tools/analyze_runs.py data/processed/2000_2026_1d_3_1/fold_2026 --runs 100 --arch attention

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.check_env_data

In [ ]:
!python -m _tools.train_rllib_pbt --population 6 --iterations 3000 --force

In [3]:
import sqlite3
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

def generate_report(base_dir_str="data/processed"):
    base_dir = Path(base_dir_str).resolve()
    all_data = []

    print(f"🔍 Сканирование директории: {base_dir}")

    # 1. Сбор данных из БД
    # rglob найдет ВСЕ базы данных во всех вложенных папках
    for db_path in base_dir.rglob("trading_factory_*.db"):
        # Относительный путь от base_dir выглядит как "CONFIG/FOLD/trading_factory.db"
        rel_path = db_path.relative_to(base_dir)
        
        # Разбираем путь на части: [CONFIG, FOLD, FILE]
        parts = rel_path.parts
        if len(parts) >= 2:
            config_name = parts[0]
            fold_name = parts[1]
            arch = db_path.name.replace("trading_factory_", "").replace(".db", "")

            try:
                conn = sqlite3.connect(db_path)
                df = pd.read_sql_query("SELECT val_loss, val_acc FROM runs WHERE status='COMPLETED'", conn)
                conn.close()
                
                if not df.empty:
                    df['config'] = config_name
                    df['fold'] = fold_name
                    df['model_type'] = arch
                    all_data.append(df)
            except Exception as e:
                print(f"⚠️ Ошибка при чтении {db_path}: {e}")

    if not all_data: return "Данные не найдены"
    df_runs = pd.concat(all_data, ignore_index=True)

    # 2. Агрегация
    stats = df_runs.groupby(['config', 'fold', 'model_type']).agg(
        models_count=('val_loss', 'count'),
        best_loss=('val_loss', 'min'),
        std_loss=('val_loss', 'std'),
        best_acc=('val_acc', 'max'),
        std_acc=('val_acc', 'std')
    ).fillna(0).reset_index()

    # 3. Добавление данных ансамбля (исправленный поиск)
    ensemble_data = []
    for json_path in base_dir.rglob("optimal_alliance.json"):
        rel_path = json_path.relative_to(base_dir)
        parts = rel_path.parts
        
        # Путь: CONFIG/FOLD/artifacts/optimal_alliance.json
        # parts: ('CONFIG', 'FOLD', 'artifacts', 'optimal_alliance.json')
        if len(parts) >= 2:
            try:
                with open(json_path, 'r') as f:
                    data = json.load(f)
                    ensemble_data.append({
                        'config': parts[0],
                        'fold': parts[1],
                        'ensemble_loss': data.get('ensemble_smoothed_loss')
                    })
            except Exception as e:
                print(f"⚠️ Ошибка чтения ансамбля {json_path}: {e}")
    
    df_ens = pd.DataFrame(ensemble_data)
    
    # 4. Слияние
    report = pd.merge(stats, df_ens, on=['config', 'fold'], how='left')
    
    # 5. Вывод
    display(HTML("<h2>🌡️ Отчет по архитектурам и стабильности</h2>"))
    style = report.style.background_gradient(subset=['best_loss', 'std_loss'], cmap='coolwarm') \
                        .background_gradient(subset=['best_acc', 'std_acc'], cmap='coolwarm_r') \
                        .format({'best_loss': '{:.4f}', 'std_loss': '{:.5f}', 
                                 'best_acc': '{:.4f}', 'std_acc': '{:.5f}', 'ensemble_loss': '{:.4f}'})
    return style

# Запуск
report = generate_report()
display(report)

🔍 Сканирование директории: /home/restorator/trader_test/data/processed


,config,fold,model_type,models_count,best_loss,std_loss,best_acc,std_acc,ensemble_loss
0,2000_2026_1d_10_1,fold_2010,attention,481,0.8682,0.02152,0.6950,0.02116,0.8285
1,2000_2026_1d_10_1,fold_2010,cnn,1323,0.8460,0.02391,0.7208,0.02348,0.8285
2,2000_2026_1d_10_1,fold_2010,conv1d+gru,132,0.8331,0.01295,0.6953,0.01228,0.8285
3,2000_2026_1d_10_1,fold_2010,mlp,168,0.8967,0.01014,0.6783,0.00937,0.8285
4,2000_2026_1d_10_1,fold_2011,attention,152,0.9190,0.00568,0.6266,0.00486,0.9020
5,2000_2026_1d_10_1,fold_2011,cnn,88,0.9115,0.00321,0.6303,0.00287,0.9020
6,2000_2026_1d_10_1,fold_2011,conv1d+gru,112,0.9116,0.00268,0.6313,0.00296,0.9020
7,2000_2026_1d_10_1,fold_2011,mlp,793,0.9373,0.00348,0.6261,0.00308,0.9020
8,2000_2026_1d_10_1,fold_2012,attention,421,0.9098,0.00347,0.6406,0.00271,0.8966
9,2000_2026_1d_10_1,fold_2012,cnn,74,0.9073,0.00307,0.6387,0.00247,0.8966
